In [2]:
import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq
from scipy.special import beta

def sinucurve(z, s, k):
    z = np.atleast_1d(z).astype(float)
    out = np.zeros_like(z)
    valid = (z > 0) & (z < 1)
    if np.any(valid):
        out[valid] = (np.sin(np.pi * z[valid]**s))**k
    return out

def sinuarea(s, k):
    if s == 1:
        # see research document
        val = (2**k / np.pi) * beta((k + 1) / 2, (k + 1) / 2)
    else:
        val, _ = quad(lambda z: (np.sin(np.pi * z**s))**k, 0, 1)
    return val

def dsinustd(z, s, k, flip=False):
    norm_const = sinuarea(s, k)
    z = np.atleast_1d(z).astype(float)
    out = np.zeros_like(z)
    if norm_const == 0:
        return out
    
    valid = (z > 0) & (z < 1)
    if not np.any(valid):
        return out
    
    z_subset = z[valid]
    z_f_subset = 1 - z_subset if flip else z_subset
    out[valid] = (np.sin(np.pi * z_f_subset**s))**k / norm_const
    return out

def psinustd(z, s, k, flip=False):
    z = np.atleast_1d(z)
    def single_p(z_val):
        if flip:
            # Recursive call mirroring R: 1 - psinustd(1 - z_val, ...)
            return 1 - psinustd(1 - z_val, s=s, k=k, flip=False)[0]
        if z_val <= 0: return 0.0
        if z_val >= 1: return 1.0
        
        # Integration of the density
        integral_result, _ = quad(dsinustd, 0, z_val, args=(s, k, False))
        return integral_result
    
    return np.array([single_p(val) for val in z])

def qsinustd(p, s, k, flip=False):
    p = np.atleast_1d(p)
    def single_q(p_val):
        if flip:
            return 1 - qsinustd(1 - p_val, s=s, k=k, flip=False)[0]
        if np.isnan(p_val) or p_val < 0 or p_val > 1: return np.nan
        if p_val == 0: return 0.0
        if p_val == 1: return 1.0
        
        # Objective function for root finding
        objective_f = lambda z: psinustd(z, s, k, flip=False)[0] - p_val
        try:
            return brentq(objective_f, 0, 1)
        except:
            return np.nan
    
    return np.array([single_q(val) for val in p])

def rsinustd(n, s, k, flip=False):
    return qsinustd(np.random.uniform(0, 1, n), s, k, flip)

def dsinu(x, a, d, s, k, flip=False):
    z = (x - a) / d
    return (1 / d) * dsinustd(z, s, k, flip)

def psinu(x, a, d, s, k, flip=False):
    z = (x - a) / d
    return psinustd(z, s, k, flip)

def qsinu(p, a, d, s, k, flip=False):
    return a + d * qsinustd(p, s, k, flip)

def rsinu(n, a, d, s, k, flip=False):
    return a + d * rsinustd(n, s, k, flip)

# --- Moments ---

def sinu_rmom(r, a, d, s, k, flip=False):
    rmom_integrand = lambda z: ((a + d * z)**r) * dsinustd(z, s, k, flip)
    val, _ = quad(rmom_integrand, 0, 1)
    return val

def sinu_mean(a, d, s, k, flip=False):
    if s == 1 or k == 0 or s == 0:
        return a + d / 2
    return a + d * sinu_rmom(1, 0, 1, s=s, k=k, flip=flip)

def sinu_cmom(r, d, s, k, flip=False):
    if s == 0 or k == 0 or s == 1:
        if r % 2 != 0: return 0.0
        flip = False
    
    stdMean = sinu_mean(0, 1, s, k, flip=flip)
    integrand = lambda z: (z - stdMean)**r * dsinustd(z, s, k, flip=flip)
    unscaledCmom, _ = quad(integrand, 0, 1)
    
    scaledCmom = (d**r) * unscaledCmom
    if flip and (r % 2 != 0):
        return -scaledCmom
    return scaledCmom

def sinu_var(d, s, k, flip=False):
    return sinu_cmom(2, d, s, k, flip=flip)

def sinu_skew(s, k, flip=False):
    if s == 0 or k == 0 or s == 1:
        return 0.0
    cmom2 = sinu_cmom(2, 1, s, k, flip=flip)
    cmom3 = sinu_cmom(3, 1, s, k, flip=flip)
    return cmom3 / (cmom2**1.5)

def sinu_kurt(s, k, flip=False):
    cmom2 = sinu_cmom(2, 1, s, k, flip=flip)
    cmom4 = sinu_cmom(4, 1, s, k, flip=flip)
    return (cmom4 / (cmom2**2)) - 3